# Day 3: heuristic baseline

Team A | Rishitha Gogineni | 21 November 2026 | `A-HEUR-01`

Compare a heuristic agent with the Day 2 random baseline, collect separate demonstration games, and test the submission package. This is the completed Colab CPU run. The heuristic won 24 of 30 evaluation battles and lost six. All 20 demonstration games and both archive tests passed without recorded invalid actions or engine rejections.

The demonstration set contains 176 decisions. The split uses 14 training games, three validation games and three held out games. No evaluation games are included in this dataset.

The agent was uploaded manually after this run. The supplied Kaggle screenshot shows Complete, with the heuristic at 489.7 and the random baseline at 103.0. These are rating snapshots. The JSON and CSV outputs retain the status saved before the upload.

This notebook does not train `A-R1` or `A-R2`. The heuristic supplies rule based labels, not expert decisions. Local checks use individual battles and the sample deck. They do not reproduce the full hosted Kaggle match harness.


## 1. Workspace and settings

Create a separate folder for this run. Use 30 battles for evaluation, 20 additional battles for demonstrations and two more for archive checks. Each phase has different game IDs and Python seeds. The native engine seed has not been verified.


In [1]:
import ast
import csv
import hashlib
import json
import platform
import random
import shutil
import subprocess
import sys
import tarfile
import traceback
import uuid
import zipfile
from datetime import datetime, timezone
from pathlib import Path

assert platform.system()=='Linux', 'Use a fresh Linux Colab CPU session.'
assert platform.machine() in ('x86_64','amd64','aarch64','arm64')
N_BATTLES=30
N_DEMO_GAMES=20
N_ARCHIVE_GAMES=2
MAX_DEMO_RECORDS_PER_GAME=500
PHASE_TIMEOUT_SECONDS=600
assert min(N_BATTLES,N_DEMO_GAMES,N_ARCHIVE_GAMES,MAX_DEMO_RECORDS_PER_GAME)>0
RUN_ID=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')+'_'+uuid.uuid4().hex[:8]
work_dir=Path('/content/ptcg_heuristic')/RUN_ID
sdk_dir=work_dir/'official_sample'
random_dir=work_dir/'random_agent'
heuristic_dir=work_dir/'heuristic_agent'
output_dir=work_dir/'outputs'
source_dir=output_dir/'source'
for folder in (sdk_dir,random_dir,heuristic_dir,source_dir):folder.mkdir(parents=True,exist_ok=False)

def save_json(path,value):
    path.parent.mkdir(parents=True,exist_ok=True)
    temporary=path.with_suffix(path.suffix+'.tmp')
    temporary.write_text(json.dumps(value,indent=2)+'\n',encoding='utf-8')
    temporary.replace(path)

def sha256(path):
    digest=hashlib.sha256()
    with path.open('rb') as handle:
        for block in iter(lambda:handle.read(1024*1024),b''):digest.update(block)
    return digest.hexdigest()

run_info={'run_id':RUN_ID,'python':sys.version,'platform':platform.platform(),'machine':platform.machine(),
          'model_id':'A-HEUR-01','unit':'individual_battle','engine_rng_verified':False,
          'requested_evaluation_battles':N_BATTLES,'requested_demo_battles':N_DEMO_GAMES,
          'requested_archive_battles':N_ARCHIVE_GAMES,'metadata_source':None}
save_json(output_dir/'run_info.json',run_info)
print('Run:',RUN_ID)
print('Python:',sys.version.split()[0],platform.system(),platform.machine())


Run: 20261121T232326Z_2121ee87
Python: 3.13.18 Linux x86_64


## 2. Official sample files

Upload the official competition ZIP used for Day 2. Record its hash and keep the sample deck unchanged. Use the bundled simulator for local tests.


In [2]:
from google.colab import files

filename='the-pokemon-company-ptcg-ai-battle-challenge-playground.zip'
archive_path=Path('/content')/filename
if not archive_path.exists():
    matches=sorted(Path('/content').glob('the-pokemon-company-ptcg-ai-battle-challenge-playground*.zip'))
    if len(matches)==1:archive_path=matches[0]
    elif matches:raise ValueError('Keep one intended competition ZIP or rename it to '+filename)
    else:
        uploaded=files.upload()
        names=[name for name in uploaded if name.lower().endswith('.zip')]
        if len(names)!=1:raise ValueError('Upload exactly one official competition ZIP.')
        archive_path=Path('/content')/names[0]
prefix='sample_submission/sample_submission/sample_submission/'
with zipfile.ZipFile(archive_path) as archive:
    names=[name for name in archive.namelist() if name.startswith(prefix) and not name.endswith('/')]
    if not names:raise ValueError('Sample package prefix not found in this ZIP version.')
    for name in names:
        relative=Path(name[len(prefix):])
        if relative.is_absolute() or '..' in relative.parts:raise ValueError('Unexpected ZIP path.')
        target=sdk_dir/relative
        target.parent.mkdir(parents=True,exist_ok=True)
        target.write_bytes(archive.read(name))
for name in ('cg/game.py','cg/sim.py','deck.csv'):assert (sdk_dir/name).is_file()
deck=[int(line) for line in (sdk_dir/'deck.csv').read_text().splitlines() if line.strip()]
assert len(deck)==60
for folder in (random_dir,heuristic_dir):shutil.copy2(sdk_dir/'deck.csv',folder/'deck.csv')
run_info['archive']={'filename':archive_path.name,'sha256':sha256(archive_path),'bytes':archive_path.stat().st_size}
run_info['deck_sha256']=sha256(sdk_dir/'deck.csv')
run_info['sdk_files']=[{'path':str(path.relative_to(sdk_dir)),'sha256':sha256(path)} for path in sorted((sdk_dir/'cg').rglob('*')) if path.is_file()]
save_json(output_dir/'run_info.json',run_info)
print('Deck entries:',len(deck))
print('Input ZIP SHA-256:',run_info['archive']['sha256'])


<IPython.core.display.HTML object>

Saving the-pokemon-company-ptcg-ai-battle-challenge-playground.zip to the-pokemon-company-ptcg-ai-battle-challenge-playground.zip
Deck entries: 60
Input ZIP SHA-256: 6fe557e77ca21b86dea9056d7f4531cbeb328abc1b25aecf9c3a4e2b7d5b2c87


## 3. Card metadata

Read printed attack damage and card HP in a separate process. Clear both mappings if the load fails and label the fallback. This run loaded 1,755 attack entries and 1,431 card HP entries. Printed damage is a rough scoring input; it does not account for attack effects, weakness or resistance.


In [3]:
METADATA_WORKER = 'import json\nfrom pathlib import Path\nimport sys\nimport traceback\n\nsys.path.insert(0, sys.argv[1])\ndestination=Path(sys.argv[2])\nresult={"status":"not_loaded","attack_damage":{},"card_hp":{}}\ntry:\n    from cg.api import all_attack, all_card_data\n    damage={int(item.attackId):int(item.damage or 0) for item in all_attack()}\n    hp={int(item.cardId):int(item.hp or 0) for item in all_card_data()}\n    result.update(status="loaded_from_official_sdk",attack_damage=damage,card_hp=hp)\nexcept Exception as error:\n    result.update(status="unavailable",error_type=type(error).__name__,error_message=str(error),traceback=traceback.format_exc())\ndestination.write_text(json.dumps(result,indent=2)+"\\n",encoding="utf-8")\n'
metadata_path=output_dir/'metadata.json'
metadata_script=source_dir/'read_metadata.py'
metadata_script.write_text(METADATA_WORKER,encoding='utf-8')

def run_process(command,log_dir,cwd):
    log_dir.mkdir(parents=True,exist_ok=True)
    status={'returncode':None,'timed_out':False,'error':None}
    with (log_dir/'stdout.txt').open('w',encoding='utf-8') as stdout, (log_dir/'stderr.txt').open('w',encoding='utf-8') as stderr:
        try:
            completed=subprocess.run(command,cwd=cwd,stdout=stdout,stderr=stderr,text=True,timeout=PHASE_TIMEOUT_SECONDS)
            status['returncode']=completed.returncode
        except subprocess.TimeoutExpired:
            status['timed_out']=True
            status['error']='Process exceeded the configured timeout; partial log files were preserved.'
        except Exception as error:
            status['error']={'type':type(error).__name__,'message':str(error),'traceback':traceback.format_exc()}
    save_json(log_dir/'process.json',status)
    return status

metadata_process=run_process([sys.executable,str(metadata_script),str(sdk_dir),str(metadata_path)],output_dir/'metadata_process',work_dir)
metadata=json.loads(metadata_path.read_text()) if metadata_path.exists() else {'status':'no_report'}
metadata_ok=metadata_process['returncode']==0 and metadata.get('status')=='loaded_from_official_sdk'
attack_damage={int(key):int(value) for key,value in metadata['attack_damage'].items()} if metadata_ok else {}
card_hp={int(key):int(value) for key,value in metadata['card_hp'].items()} if metadata_ok else {}
run_info['metadata_source']=metadata.get('status') if metadata_ok else 'option_only_fallback'
run_info['attack_entries']=len(attack_damage)
run_info['card_entries']=len(card_hp)
save_json(output_dir/'run_info.json',run_info)
print('Metadata:',run_info['metadata_source'])
print('Attack entries:',len(attack_damage),'| Card HP entries:',len(card_hp))


Metadata: loaded_from_official_sdk
Attack entries: 1755 | Card HP entries: 1431


## 4. Random and heuristic agents

Keep the accepted random policy unchanged. Save the heuristic separately as `source/heuristic.py`. Its scoring rules are assumptions tested in the following games. The exported policy includes card metadata used by those rules.


In [4]:
RANDOM_SOURCE = '"""Team A random baseline. Uses the official sample deck and maxCount policy."""\n\nimport os\nimport random\n\n\ndef read_deck(configuration=None):\n    candidates = []\n    raw_path = (configuration or {}).get("__raw_path__")\n    if raw_path:\n        candidates.append(os.path.join(os.path.dirname(os.path.abspath(raw_path)), "deck.csv"))\n    module_path = globals().get("__file__")\n    if module_path:\n        candidates.append(os.path.join(os.path.dirname(os.path.abspath(module_path)), "deck.csv"))\n    candidates.extend(["/kaggle_simulations/agent/deck.csv", "deck.csv"])\n    for path in candidates:\n        if os.path.isfile(path):\n            with open(path, encoding="utf-8") as handle:\n                deck = [int(line.strip()) for line in handle if line.strip()]\n            if len(deck) != 60:\n                raise ValueError("deck.csv must contain exactly 60 card IDs.")\n            return deck\n    raise FileNotFoundError("deck.csv was not found beside the agent or in the Kaggle agent folder.")\n\n\ndef agent(observation, configuration=None):\n    selection = observation.get("select")\n    if selection is None:\n        return read_deck(configuration)\n    options = selection["option"]\n    minimum = selection["minCount"]\n    maximum = selection["maxCount"]\n    if not (0 <= minimum <= maximum <= len(options)):\n        raise ValueError("Invalid option counts in the observation.")\n    return random.sample(range(len(options)), maximum)\n'
HEURISTIC_TEMPLATE = '"""Team A: rule-based action-selection baseline, with no learned model.\nThe available options are already legal. Only their indices are returned.\n"""\nimport os\n\n# Values from cabt\'s documented OptionType and SelectContext enums.\nSELECT_MAIN = 0\nSELECT_CARD = 1\nSELECT_ENERGY = 4\nSELECT_COUNT = 8\nSELECT_YES_NO = 9\nTYPE_NUMBER = 0\nTYPE_YES = 1\nTYPE_NO = 2\nTYPE_CARD = 3\nTYPE_ENERGY = 6\nTYPE_PLAY = 7\nTYPE_ATTACH = 8\nTYPE_EVOLVE = 9\nTYPE_ABILITY = 10\nTYPE_DISCARD = 11\nTYPE_RETREAT = 12\nTYPE_ATTACK = 13\nTYPE_END = 14\nAREA_HAND = 2\nAREA_ACTIVE = 4\nAREA_BENCH = 5\n\n# Generated from the official local SDK. These are base attack damage values,\n# not guaranteed effective damage after weakness, resistance or effects.\nATTACK_DAMAGE = __ATTACK_DAMAGE__\nCARD_HP = __CARD_HP__\n\n\ndef read_deck(configuration=None):\n    paths = []\n    raw_path = (configuration or {}).get(\'__raw_path__\')\n    if raw_path:\n        paths.append(os.path.join(os.path.dirname(os.path.abspath(raw_path)), \'deck.csv\'))\n    module_path = globals().get(\'__file__\')\n    if module_path:\n        paths.append(os.path.join(os.path.dirname(os.path.abspath(module_path)), \'deck.csv\'))\n    paths.extend([\'/kaggle_simulations/agent/deck.csv\', \'deck.csv\'])\n    for path in paths:\n        if os.path.isfile(path):\n            with open(path, encoding=\'utf-8\') as handle:\n                ids = [int(s.strip()) for s in handle if s.strip()]\n            if len(ids) != 60:\n                raise ValueError(\'deck.csv must contain 60 card IDs\')\n            return ids\n    raise FileNotFoundError(\'deck.csv not found\')\n\n\ndef _player_state(obs):\n    state = obs.get(\'current\') or {}\n    players = state.get(\'players\') or []\n    own_idx = state.get(\'yourIndex\', 0)\n    if own_idx not in (0, 1) or len(players) < 2:\n        return state, {}, {}, 0\n    return state, players[own_idx] or {}, players[1 - own_idx] or {}, own_idx\n\n\ndef _active(player):\n    active = player.get(\'active\') or []\n    return active[0] if active and isinstance(active[0], dict) else {}\n\n\ndef _card_for_option(obs, option):\n    state, mine, theirs, who = _player_state(obs)\n    owner = option.get(\'playerIndex\', who)\n    players = (state.get(\'players\') or [])\n    if type(owner) is not int or not 0 <= owner < len(players):\n        return {}\n    player = players[owner] or {}\n    area = option.get(\'area\')\n    index = option.get(\'index\')\n    if type(index) is not int or index < 0:\n        return {}\n    name = {1: \'deck\', 2: \'hand\', 3: \'discard\', 4: \'active\', 5: \'bench\', 6: \'prize\'}.get(area)\n    values = player.get(name) if name else None\n    if isinstance(values, list) and index < len(values) and isinstance(values[index], dict):\n        return values[index]\n    return {}\n\n\ndef _main_score(option, obs):\n    typ = option.get(\'type\')\n    state, mine, theirs, who = _player_state(obs)\n    turn_actions = state.get(\'turnActionCount\', 0) or 0\n    if typ == TYPE_ATTACK:\n        damage = ATTACK_DAMAGE.get(option.get(\'attackId\'), 0)\n        opp_active = _active(theirs)\n        hp = opp_active.get(\'hp\')\n        # Knockout check uses only printed base damage: it is a rough signal.\n        knockout = (type(hp) in (int, float) and hp > 0 and damage >= hp)\n        return 220 + damage / 10 if knockout else 92 + min(damage / 10, 35) + min(turn_actions, 30) / 2\n    if typ == TYPE_EVOLVE:\n        return 112\n    if typ == TYPE_ATTACH:\n        # Prefer energy on the active Pokémon, unless it is already attached.\n        target = option.get(\'inPlayArea\')\n        return 116 if target == AREA_ACTIVE else 104\n    if typ == TYPE_PLAY:\n        # Legal plays develop the board, but the score is below a clear KO.\n        bench_count = len(mine.get(\'bench\') or [])\n        return 108 if bench_count < 2 else 94\n    if typ == TYPE_ABILITY:\n        return 96\n    if typ == TYPE_RETREAT:\n        own_active = _active(mine)\n        hp, max_hp = own_active.get(\'hp\'), own_active.get(\'maxHp\')\n        if (type(hp) in (int, float) and type(max_hp) in (int, float)\n                and max_hp > 0 and hp < max_hp * .30 and mine.get(\'bench\')):\n            return 120\n        return 10\n    if typ == TYPE_DISCARD:\n        return 12\n    if typ == TYPE_END:\n        return -100\n    return 25\n\n\ndef _option_score(option, obs):\n    selection = obs.get(\'select\') or {}\n    selection_type = selection.get(\'type\')\n    context = selection.get(\'context\')\n    typ = option.get(\'type\')\n    if selection_type == SELECT_MAIN:\n        return _main_score(option, obs)\n    if selection_type == SELECT_YES_NO:\n        if context == 42:  # Mulligan: prefer keeping a playable hand.\n            return 10 if typ == TYPE_NO else 0\n        if context in (43, 44):  # Optional activation/effect.\n            return 10 if typ == TYPE_YES else 0\n        return 2 if typ == TYPE_YES else 1\n    if selection_type == SELECT_COUNT:\n        return float(option.get(\'number\', 0) or 0)\n    if selection_type == SELECT_ENERGY:\n        # During energy discards, spend bench energy first if available.\n        if context == 30:\n            return 5 if option.get(\'area\') == AREA_BENCH else 0\n        return 5 if option.get(\'area\') == AREA_ACTIVE else 2\n    if selection_type == SELECT_CARD:\n        card = _card_for_option(obs, option)\n        cid = card.get(\'id\') or option.get(\'cardId\')\n        hp = CARD_HP.get(cid, 0)\n        if context in (1, 2, 3, 4, 5, 6, 18, 19):\n            return hp / 20.0  # Setup/switch: favor sturdier Pokémon.\n        if context == 8:  # Discard: preserve higher-HP Pokémon if possible.\n            return -hp / 20.0\n        if context in (13, 14, 15):  # Damage target: favor opponent targets.\n            current = obs.get(\'current\') or {}\n            return 10 if option.get(\'playerIndex\') != current.get(\'yourIndex\') else 0\n        if context in (21, 22):  # Attachment target: generally favor Active.\n            return 10 if option.get(\'area\') == AREA_ACTIVE else 0\n        return 0\n    return 0\n\n\ndef agent(observation, configuration=None):\n    selection = observation.get(\'select\')\n    if selection is None:\n        return read_deck(configuration)\n    options = selection.get(\'option\') or []\n    minimum = selection[\'minCount\']\n    maximum = selection[\'maxCount\']\n    if not 0 <= minimum <= maximum <= len(options):\n        raise ValueError(\'Invalid selection bounds\')\n    # Match the accepted Day 2 behavior for number of selections.\n    # Stable sort makes runs easier to debug. This is not a search policy.\n    ranked = sorted(range(len(options)),\n                    key=lambda i: (-_option_score(options[i], observation), i))\n    return ranked[:maximum]\n'
HEURISTIC_SOURCE=HEURISTIC_TEMPLATE.replace('__ATTACK_DAMAGE__',repr(attack_damage)).replace('__CARD_HP__',repr(card_hp))
for value in (RANDOM_SOURCE,HEURISTIC_SOURCE):ast.parse(value)
(random_dir/'main.py').write_text(RANDOM_SOURCE,encoding='utf-8')
(heuristic_dir/'main.py').write_text(HEURISTIC_SOURCE,encoding='utf-8')
(source_dir/'random_baseline_frozen.py').write_text(RANDOM_SOURCE,encoding='utf-8')
(source_dir/'heuristic.py').write_text(HEURISTIC_SOURCE,encoding='utf-8')
assert sha256(random_dir/'main.py')=='9f447d9252f659659df0748c11734a188ce8e306a517a8ec842f1a79d950a4ff'
run_info['frozen_random_sha256']=sha256(random_dir/'main.py')
run_info['heuristic_sha256']=sha256(heuristic_dir/'main.py')
save_json(output_dir/'run_info.json',run_info)
print('Frozen random and separate heuristic source saved.')


Frozen random and separate heuristic source saved.


## 5. Evaluation worker

Save results after every completed battle. Write the current game ID before each battle and flush each accepted demonstration record to disk. A timeout or native crash can leave partial records for diagnosis.

Check deck loading without `__file__`, single and multiple selections, optional choices and empty choices. Alternate agent seats and vary the simulator player setting.


In [5]:
RUNNER_SOURCE = '"""Reusable direct-SDK individual-battle runner. SDK must be importable as cg."""\n\nimport copy\nimport random\nimport time\nimport traceback\n\n\ndef check_action(action, selection):\n    if not isinstance(action, list) or not all(type(index) is int for index in action):\n        raise TypeError("The agent must return a list of integer option indices.")\n    if not selection["minCount"] <= len(action) <= selection["maxCount"]:\n        raise ValueError("Action length violates selection counts.")\n    if len(set(action)) != len(action):\n        raise ValueError("Repeated option indices.")\n    if any(index < 0 or index >= len(selection["option"]) for index in action):\n        raise ValueError("Option index out of range.")\n\n\ndef run_battle(agents, seed, max_steps=10000, reverse_player=False, decision_observer=None):\n    from cg.game import battle_start, battle_select, battle_finish\n    from cg.sim import Battle\n\n    random.seed(seed)\n    started = time.perf_counter()\n    record = {\n        "python_seed": seed, "unit": "individual_battle", "status": "running",\n        "finished": False, "result_code": None, "steps": 0,\n        "reverse_player": reverse_player, "seconds": None,\n        "max_decision_ms": 0.0, "selection_types": [],\n        "invalid_responses": 0, "engine_rejections": 0,\n        "error_type": "", "error_message": "", "traceback": "", "cleanup_error": "",\n    }\n    seen_types = set()\n    try:\n        initial = {"select": None, "logs": [], "current": None, "search_begin_input": None}\n        decks = [callback(dict(initial)) for callback in agents]\n        for deck in decks:\n            if not isinstance(deck, list) or len(deck) != 60 or not all(type(card) is int for card in deck):\n                raise ValueError("Initial callback did not return 60 integer card IDs.")\n        observation, start = battle_start(decks[0], decks[1], reverse_player)\n        if observation is None:\n            raise RuntimeError(f"Battle start error: player={start.errorPlayer}, type={start.errorType}")\n        for step in range(max_steps + 1):\n            current = observation.get("current")\n            if current is None:\n                raise ValueError("Missing current state in direct battle API.")\n            result_code = current["result"]\n            if result_code != -1:\n                record.update(status="passed", finished=True, result_code=result_code)\n                break\n            if step == max_steps:\n                record.update(status="step_limit", error_type="StepLimit", error_message=f"No terminal result after {max_steps} selections.")\n                break\n            player = current["yourIndex"]\n            if player not in (0, 1):\n                raise ValueError("Unexpected acting player.")\n            selection = observation.get("select")\n            if selection is None:\n                raise ValueError("Missing selection during the battle.")\n            seen_types.add(selection["type"])\n            t0 = time.perf_counter()\n            action = agents[player](observation)\n            record["max_decision_ms"] = max(record["max_decision_ms"], (time.perf_counter() - t0) * 1000)\n            try:\n                check_action(action, selection)\n            except Exception:\n                record["invalid_responses"] += 1\n                raise\n            snapshot = None\n            if decision_observer is not None:\n                snapshot = {key: copy.deepcopy(observation[key]) for key in ("logs", "current", "select") if key in observation}\n            try:\n                observation = battle_select(action)\n            except Exception:\n                record["engine_rejections"] += 1\n                raise\n            record["steps"] = step + 1\n            if decision_observer is not None:\n                decision_observer(snapshot, list(action), step, player)\n    except Exception as error:\n        record.update(status="error", error_type=type(error).__name__, error_message=str(error), traceback=traceback.format_exc())\n    finally:\n        try:\n            if Battle.battle_ptr:\n                battle_finish()\n        except Exception as error:\n            record["cleanup_error"] = f"{type(error).__name__}: {error}"\n            record["status"] = "cleanup_error"\n        finally:\n            Battle.battle_ptr = None\n            record["seconds"] = round(time.perf_counter() - started, 6)\n            record["max_decision_ms"] = round(record["max_decision_ms"], 6)\n            record["selection_types"] = sorted(seen_types)\n    return record\n'
EVALUATION_WORKER = '"""Individual-battle comparison, incremental records and separate teacher demonstrations."""\nimport argparse\nfrom collections import Counter\nimport csv\nimport json\nimport math\nfrom pathlib import Path\nimport sys\nimport traceback\n\n\ndef save_json(path,value):\n    temporary=path.with_suffix(path.suffix+".tmp")\n    temporary.write_text(json.dumps(value,indent=2)+"\\n",encoding="utf-8")\n    temporary.replace(path)\n\n\ndef load_agent(directory):\n    path=(directory/"main.py").resolve()\n    namespace={}\n    exec(compile(path.read_text(encoding="utf-8"),str(path),"exec"),namespace)\n    functions=[value for value in namespace.values() if callable(value)]\n    if not functions or functions[-1].__name__!="agent":\n        raise ValueError("Expected agent as the last callable.")\n    function=functions[-1]\n    config={"__raw_path__":str(path)}\n    return lambda obs:function(obs,config),namespace\n\n\ndef wilson(wins,total):\n    if not total:return None\n    z=1.959963984540054\n    p=wins/total\n    denominator=1+z*z/total\n    center=(p+z*z/(2*total))/denominator\n    half=z*math.sqrt(p*(1-p)/total+z*z/(4*total*total))/denominator\n    return [max(0.0,center-half),min(1.0,center+half)]\n\n\ndef main():\n    parser=argparse.ArgumentParser()\n    for flag in ("sdk-dir","random-dir","heuristic-dir","output-dir","run-id","phase"):\n        parser.add_argument("--"+flag,required=True)\n    parser.add_argument("--games",type=int,default=30)\n    parser.add_argument("--seed-start",type=int,default=7000)\n    parser.add_argument("--collect",action="store_true")\n    parser.add_argument("--max-demo-records-per-game",type=int,default=500)\n    args=parser.parse_args()\n    if args.games<=0 or args.max_demo_records_per_game<=0:\n        parser.error("Games and record cap must be positive.")\n    root=Path(args.output_dir).resolve()\n    root.mkdir(parents=True,exist_ok=True)\n    sys.path.insert(0,str(Path(args.sdk_dir).resolve()))\n    from heuristic_runner import run_battle,check_action\n    rows=[]\n    contracts=[]\n    report={"status":"running","all_passed":False,"requested_battles":args.games,\n            "run_id":args.run_id,"phase":args.phase,"unit":"individual_battle",\n            "contract_checks":contracts,"battles":rows,"current_game_id":None,\n            "demonstrations_enabled":args.collect,"error":None}\n    fields=["game_id","battle_id","python_seed","heuristic_seat","reverse_player","outcome",\n            "status","finished","result_code","steps","seconds","max_decision_ms",\n            "invalid_responses","engine_rejections","demo_records","supported_decisions_seen",\n            "demo_cap_reached","demo_file","error_type","error_message","cleanup_error"]\n\n    def checkpoint():\n        save_json(root/"report.json",report)\n        temporary=root/"battle_results.csv.tmp"\n        with temporary.open("w",newline="",encoding="utf-8") as handle:\n            writer=csv.DictWriter(handle,fieldnames=fields,extrasaction="ignore")\n            writer.writeheader();writer.writerows(rows)\n        temporary.replace(root/"battle_results.csv")\n\n    checkpoint()\n    try:\n        heuristic,heuristic_namespace=load_agent(Path(args.heuristic_dir))\n        random_agent,_=load_agent(Path(args.random_dir))\n        initial={"select":None,"logs":[],"current":None,"search_begin_input":None}\n        expected=[int(line) for line in (Path(args.heuristic_dir)/"deck.csv").read_text().splitlines() if line.strip()]\n        assert len(expected)==60 and heuristic(initial)==expected\n        contracts.append({"case":"deck_callback_without___file__","passed":True})\n        cases=[\n            ("single",{"minCount":1,"maxCount":1,"option":[{"type":14},{"type":13,"attackId":1}],"type":0,"context":0}),\n            ("multiple",{"minCount":2,"maxCount":2,"option":[{"type":3}]*3,"type":1,"context":2}),\n            ("zero",{"minCount":0,"maxCount":0,"option":[],"type":4,"context":30}),\n            ("optional",{"minCount":0,"maxCount":1,"option":[{"type":1},{"type":2}],"type":9,"context":43}),\n        ]\n        for name,selection in cases:\n            check_action(heuristic({"select":selection,"current":None}),selection)\n            contracts.append({"case":name,"passed":True})\n        checkpoint()\n        for index in range(args.games):\n            game_id=f"{args.run_id}:{args.phase}:{index+1:04d}"\n            report["current_game_id"]=game_id\n            checkpoint()\n            print("Starting",game_id,flush=True)\n            seat=index%2\n            agents=[heuristic,random_agent] if seat==0 else [random_agent,heuristic]\n            counts={"saved":0,"seen":0}\n            relative=f"records/game_{index+1:04d}.jsonl" if args.collect else ""\n            demo_handle=None\n            if args.collect:\n                (root/"records").mkdir(exist_ok=True)\n                demo_handle=(root/relative).open("w",encoding="utf-8")\n\n            def observe(observation,action,step,player):\n                selection=observation.get("select") or {}\n                if player!=seat or selection.get("type")!=0 or selection.get("minCount")!=1 or selection.get("maxCount")!=1:\n                    return\n                counts["seen"]+=1\n                if counts["saved"]>=args.max_demo_records_per_game:return\n                options=selection["option"]\n                scores=[float(heuristic_namespace["_option_score"](option,observation)) for option in options]\n                assert len(action)==1 and all(math.isfinite(score) for score in scores)\n                assert scores[action[0]]==max(scores)\n                example={"schema_version":"heuristic-main-v1","run_id":args.run_id,"game_id":game_id,\n                         "step":step,"turn":observation["current"].get("turn"),"acting_player":player,\n                         "teacher_id":"A-HEUR-01","teacher_kind":"handcrafted_rules_not_expert",\n                         "selection_type":selection["type"],"selection_context":selection.get("context"),\n                         "legal_option_indices":list(range(len(options))),"chosen_option_index":action[0],\n                         "chosen_option_indices":action,"teacher_scores":scores,"selection_accepted":True,\n                         "observation":observation}\n                demo_handle.write(json.dumps(example)+"\\n")\n                demo_handle.flush()\n                counts["saved"]+=1\n\n            try:\n                result=run_battle(agents,seed=args.seed_start+index,reverse_player=bool((index//2)%2),\n                                  decision_observer=observe if args.collect else None)\n            finally:\n                if demo_handle is not None:demo_handle.close()\n            win_code=result["result_code"]\n            if result["status"]!="passed" or not result["finished"]:outcome="error"\n            elif win_code==2:outcome="draw"\n            elif win_code==seat:outcome="win"\n            elif win_code in (0,1):outcome="loss"\n            else:outcome="unknown_result"\n            result.update(game_id=game_id,battle_id=index+1,heuristic_seat=seat,outcome=outcome,\n                          demo_records=counts["saved"],supported_decisions_seen=counts["seen"],\n                          demo_cap_reached=counts["seen"]>counts["saved"],demo_file=relative)\n            rows.append(result)\n            checkpoint()\n            print(f"Finished {index+1}/{args.games}: {outcome}; status={result[\'status\']}; samples={counts[\'saved\']}",flush=True)\n        report["all_passed"]=(len(rows)==args.games and all(row["status"]=="passed" and row["finished"] and row["outcome"] in ("win","loss","draw") for row in rows))\n        report["status"]="passed" if report["all_passed"] else "failed"\n    except Exception as error:\n        report.update(status="error",error={"type":type(error).__name__,"message":str(error),"traceback":traceback.format_exc()})\n    finally:\n        checkpoint()\n        counts=Counter(row["outcome"] for row in rows)\n        valid=counts["win"]+counts["loss"]+counts["draw"]\n        summary={"phase":args.phase,"requested_battles":args.games,"recorded_battles":len(rows),\n                 "valid_battles":valid,"win":counts["win"],"loss":counts["loss"],"draw":counts["draw"],\n                 "error":counts["error"],"unknown_result":counts["unknown_result"],\n                 "win_rate":counts["win"]/valid if valid else None,\n                 "score_rate":(counts["win"]+.5*counts["draw"])/valid if valid else None,\n                 "win_rate_wilson_95":wilson(counts["win"],valid),\n                 "local_checks_passed":report["all_passed"],"unit":"individual_battle",\n                 "demonstration_records":sum(row["demo_records"] for row in rows),\n                 "note":"Rates use completed valid battles. Do not claim improvement if any game failed. The interval is an approximate binomial interval; it does not account for opponent/deck dependence or native RNG control."}\n        save_json(root/"summary.json",summary)\n    print(json.dumps(summary,indent=2),flush=True)\n    return 0 if report["all_passed"] else 1\n\n\nif __name__=="__main__":\n    raise SystemExit(main())\n'
for value in (RUNNER_SOURCE,EVALUATION_WORKER):ast.parse(value)
(source_dir/'heuristic_runner.py').write_text(RUNNER_SOURCE,encoding='utf-8')
worker_path=source_dir/'evaluate_heuristic.py'
worker_path.write_text(EVALUATION_WORKER,encoding='utf-8')

def run_phase(phase,agent_directory,games,seed_start,collect=False):
    phase_dir=output_dir/phase
    working_dir=work_dir/(phase+'_working_directory')
    working_dir.mkdir(parents=True,exist_ok=False)
    command=[sys.executable,str(worker_path),'--sdk-dir',str(sdk_dir),
             '--random-dir',str(random_dir),'--heuristic-dir',str(agent_directory),
             '--output-dir',str(phase_dir),'--run-id',RUN_ID,'--phase',phase,
             '--games',str(games),'--seed-start',str(seed_start),
             '--max-demo-records-per-game',str(MAX_DEMO_RECORDS_PER_GAME)]
    if collect:command.append('--collect')
    process=run_process(command,phase_dir,working_dir)
    report_path=phase_dir/'report.json'
    summary_path=phase_dir/'summary.json'
    report=json.loads(report_path.read_text()) if report_path.exists() else {'status':'no_report','all_passed':False,'battles':[]}
    summary=json.loads(summary_path.read_text()) if summary_path.exists() else None
    passed=process['returncode']==0 and report.get('all_passed') is True and len(report.get('battles',[]))==games
    print(phase,'passed:',passed)
    if summary:print(json.dumps(summary,indent=2))
    if not passed:
        print('Partial results and process logs:',phase_dir)
        print((phase_dir/'stderr.txt').read_text()[-2500:])
    return passed,report,summary


## 6. Compare the agents

Play 30 individual battles against the frozen random baseline. Count wins, losses, draws and errors separately. The heuristic recorded 24 wins, six losses, no draws and no errors, giving an observed win rate of 80 percent.

The approximate Wilson interval is 62.7 to 90.5 percent. It describes the binary win fraction among valid battles, not the score rate that gives half a point for draws. Shared decks and uncontrolled native randomness limit the interpretation. This comparison does not establish a stable Kaggle rating.


In [6]:
evaluation_passed,evaluation_report,evaluation_summary=run_phase('evaluation',heuristic_dir,N_BATTLES,7000)


evaluation passed: True
{
  "phase": "evaluation",
  "requested_battles": 30,
  "recorded_battles": 30,
  "valid_battles": 30,
  "win": 24,
  "loss": 6,
  "draw": 0,
  "error": 0,
  "unknown_result": 0,
  "win_rate": 0.8,
  "score_rate": 0.8,
  "win_rate_wilson_95": [
    0.6269430358685175,
    0.9049489282271013
  ],
  "local_checks_passed": true,
  "unit": "individual_battle",
  "demonstration_records": 0,
  "note": "Rates use completed valid battles. Do not claim improvement if any game failed. The interval is an approximate binomial interval; it does not account for opponent/deck dependence or native RNG control."
}


## 7. Collect demonstrations

Use separate games to collect the heuristic player's accepted MAIN decisions that require one option. Each record contains the visible observation, legal options, selected option, heuristic scores, step and game ID. Exclude the opaque search payload. Future outcomes are not input features.

All 20 collection games passed and produced 176 records. The records are split by game: 127 training examples, 28 validation examples and 21 held out examples. Fit preprocessing on training games only. Keep the held out games for the final check.

These labels reproduce handcrafted preferences. They are not optimal action values or complete reinforcement learning transitions. The collection limit is 500 rows per game; any truncation is recorded.


In [7]:
demonstrations_passed=False
demonstration_report={'battles':[]}
demonstration_summary=None
if evaluation_passed:
    demonstrations_passed,demonstration_report,demonstration_summary=run_phase('demonstrations',heuristic_dir,N_DEMO_GAMES,9000,collect=True)
else:
    print('Demonstrations skipped until the evaluation executes without errors.')

eligible=[]
evaluation_ids={row['game_id'] for row in evaluation_report.get('battles',[])}
for battle in demonstration_report.get('battles',[]):
    if battle['status']!='passed' or not battle['finished'] or battle.get('demo_records',0)==0:continue
    path=output_dir/'demonstrations'/battle['demo_file']
    count=0
    with path.open(encoding='utf-8') as handle:
        for line in handle:
            row=json.loads(line)
            assert row['game_id']==battle['game_id'] and row['game_id'] not in evaluation_ids
            assert row['acting_player']==row['observation']['current']['yourIndex']
            assert set(row['observation'])<={'logs','current','select'}
            assert row['selection_type']==0 and row['selection_accepted']
            assert row['observation']['select']['minCount']==row['observation']['select']['maxCount']==1
            assert len(row['teacher_scores'])==len(row['observation']['select']['option'])
            count+=1
    assert count==battle['demo_records']
    eligible.append({'game_id':battle['game_id'],'path':'demonstrations/'+battle['demo_file'],'rows':count})
eligible.sort(key=lambda item:item['game_id'])
random.Random(2026).shuffle(eligible)
if len(eligible)>=3:
    heldout_count=max(1,round(len(eligible)*0.15))
    validation_count=max(1,round(len(eligible)*0.15))
    heldout=eligible[:heldout_count]
    validation=eligible[heldout_count:heldout_count+validation_count]
    training=eligible[heldout_count+validation_count:]
else:
    training,validation,heldout=eligible,[],[]
manifest={
    'schema_version':'heuristic-main-v1','teacher_id':'A-HEUR-01','teacher_kind':'handcrafted_rules_not_expert',
    'heuristic_sha256':sha256(heuristic_dir/'main.py'),'metadata_source':run_info['metadata_source'],
    'supported_context':'single-choice MAIN only','split_seed':2026,'grouping_key':'game_id',
    'train':training,'validation':validation,'held_out':heldout,
    'eligible_games':len(eligible),'eligible_rows':sum(item['rows'] for item in eligible),
    'split_ready':bool(training and validation and heldout),'collection_passed':demonstrations_passed,
    'excluded_game_ids':[battle['game_id'] for battle in demonstration_report.get('battles',[]) if battle['game_id'] not in {item['game_id'] for item in eligible}],
    'evaluation_games_excluded_from_training':sorted(evaluation_ids),
    'notes':'Fit preprocessing on training games only. Preserve held-out games; do not tune against them. Labels reproduce this teacher, not optimal play. Rows are decision examples, not complete RL transitions.'
}
save_json(output_dir/'dataset_manifest.json',manifest)
print('Eligible demonstration games:',manifest['eligible_games'],'| rows:',manifest['eligible_rows'])
print('Train / validation / held-out games:',len(training),len(validation),len(heldout))
print('Dataset split ready:',manifest['split_ready'])


demonstrations passed: True
{
  "phase": "demonstrations",
  "requested_battles": 20,
  "recorded_battles": 20,
  "valid_battles": 20,
  "win": 20,
  "loss": 0,
  "draw": 0,
  "error": 0,
  "unknown_result": 0,
  "win_rate": 1.0,
  "score_rate": 1.0,
  "win_rate_wilson_95": [
    0.8388748419471806,
    1.0
  ],
  "local_checks_passed": true,
  "unit": "individual_battle",
  "demonstration_records": 176,
  "note": "Rates use completed valid battles. Do not claim improvement if any game failed. The interval is an approximate binomial interval; it does not account for opponent/deck dependence or native RNG control."
}
Eligible demonstration games: 20 | rows: 176
Train / validation / held-out games: 14 3 3
Dataset split ready: True


## 8. Build and test the archive

Package the agent after evaluation and demonstration checks pass. Check the archive members and hashes, extract it into a new directory, and play two additional battles in another process. Both tests passed. The ready archive contains `main.py` and `deck.csv` and is 12,504 bytes.


In [8]:
package_ready=False
archive_checks_passed=False
package_manifest={'built':False,'package_ready':False,'kaggle_status':'not_submitted'}
submission_path=output_dir/'submission.tar.gz'
candidate_path=work_dir/'candidate_submission.tar.gz'
if evaluation_passed and demonstrations_passed:
    try:
        with tarfile.open(candidate_path,'w:gz') as archive:
            for name in ('main.py','deck.csv'):archive.add(heuristic_dir/name,arcname=name,recursive=False)
        unpacked=work_dir/'unpacked_submission'
        unpacked.mkdir()
        with tarfile.open(candidate_path,'r:gz') as archive:
            members=archive.getmembers()
            assert len(members)==2 and {item.name for item in members}=={'main.py','deck.csv'}
            for item in members:
                assert item.isfile() and not item.issym() and not item.islnk()
                content=archive.extractfile(item).read()
                assert content==(heuristic_dir/item.name).read_bytes()
                (unpacked/item.name).write_bytes(content)
        assert candidate_path.stat().st_size<=int(197.7*1024*1024)
        package_manifest.update(built=True,sha256=sha256(candidate_path),bytes=candidate_path.stat().st_size,
                                files={name:sha256(unpacked/name) for name in ('main.py','deck.csv')})
        archive_checks_passed,archive_report,archive_summary=run_phase('archive_checks',unpacked,N_ARCHIVE_GAMES,11000)
        if archive_checks_passed:
            shutil.copy2(candidate_path,submission_path)
            assert sha256(submission_path)==package_manifest['sha256']
            package_ready=True
    except Exception as error:
        package_manifest['error']={'type':type(error).__name__,'message':str(error),'traceback':traceback.format_exc()}
        print('Package check failed:',type(error).__name__,str(error))
else:
    print('Package skipped because a required local phase did not pass.')
package_manifest['package_ready']=package_ready
package_manifest['archive_checks_passed']=archive_checks_passed
save_json(output_dir/'package_manifest.json',package_manifest)
print('Package ready:',package_ready)
if package_ready:print('Archive:',submission_path.name,'| SHA-256:',package_manifest['sha256'])


archive_checks passed: True
{
  "phase": "archive_checks",
  "requested_battles": 2,
  "recorded_battles": 2,
  "valid_battles": 2,
  "win": 2,
  "loss": 0,
  "draw": 0,
  "error": 0,
  "unknown_result": 0,
  "win_rate": 1.0,
  "score_rate": 1.0,
  "win_rate_wilson_95": [
    0.34238022750665303,
    1.0
  ],
  "local_checks_passed": true,
  "unit": "individual_battle",
  "demonstration_records": 0,
  "note": "Rates use completed valid battles. Do not claim improvement if any game failed. The interval is an approximate binomial interval; it does not account for opponent/deck dependence or native RNG control."
}
Package ready: True
Archive: submission.tar.gz | SHA-256: 4cc134e6e96bc6878d7a0c90bb862a5be55c30dd9fc1f86ad9aef10fceb10c09


## 9. Experiment record

Save the actual results, package hash and status at the end of the run. The saved outputs say `not_submitted` because the manual Kaggle upload happened afterward. The submission screenshot and later record are stored separately. The screenshot does not show the submission ID, absolute observation time or rating uncertainty.


In [9]:
summary=evaluation_summary or {}
record={
    'team':'GROUP_PROGRAMMING_09_a','member':'Rishitha Gogineni','run_id':RUN_ID,'model_id':'A-HEUR-01',
    'method':'rule-based heuristic; no model training','opponent':'frozen Day 2 random baseline',
    'heuristic_sha256':sha256(heuristic_dir/'main.py'),'deck_sha256':sha256(heuristic_dir/'deck.csv'),
    'evaluation_battles_requested':N_BATTLES,'evaluation_battles_recorded':summary.get('recorded_battles'),
    'wins':summary.get('win'),'losses':summary.get('loss'),'draws':summary.get('draw'),'errors':summary.get('error'),
    'win_rate':summary.get('win_rate'),'score_rate':summary.get('score_rate'),
    'evaluation_passed':evaluation_passed,'demonstrations_passed':demonstrations_passed,
    'eligible_demo_rows':manifest['eligible_rows'],'archive_checks_passed':archive_checks_passed,
    'package_ready':package_ready,'submission_id':'','kaggle_status':'not_submitted','kaggle_rating':'',
}
with (output_dir/'experiment_log.csv').open('w',newline='',encoding='utf-8') as handle:
    writer=csv.DictWriter(handle,fieldnames=list(record));writer.writeheader();writer.writerow(record)
status={key:record[key] for key in ('model_id','run_id','evaluation_passed','demonstrations_passed','archive_checks_passed','package_ready','kaggle_status')}
status['dataset_split_ready']=manifest['split_ready']
save_json(output_dir/'run_status.json',status)
save_json(output_dir/'run_info.json',run_info)
(output_dir/'README.txt').write_text(
    'HEURISTIC EXPERIMENT\n\n'
    'source/heuristic.py is a separate rule-based candidate. Do not overwrite the accepted random baseline.\n'
    'evaluation/ contains head-to-head results; demonstrations/ contains separately generated teacher examples.\n'
    'dataset_manifest.json defines game-level train, validation and held-out groups.\n'
    'Use only passed eligible games in the manifest. Partial failed-game rows are diagnostics.\n'
    'source/heuristic_runner.py adds an optional observation hook to the Day 2 battle logic.\n'
    'If package_ready is true, submission.tar.gz passed a fresh-extraction process test.\n'
    'Upload only that inner archive to Kaggle, not the output ZIP or notebook.\n'
    'No trained linear or MLP model is produced by this notebook.\n',encoding='utf-8')
print(json.dumps(status,indent=2))


{
  "model_id": "A-HEUR-01",
  "run_id": "20261121T232326Z_2121ee87",
  "evaluation_passed": true,
  "demonstrations_passed": true,
  "archive_checks_passed": true,
  "package_ready": true,
  "kaggle_status": "not_submitted",
  "dataset_split_ready": true
}


## 10. Save outputs

Download the results ZIP and save this executed notebook from Colab. The ZIP contains source files, results, logs, demonstration records and the split manifest. Upload only the inner `submission.tar.gz` to Kaggle when the package checks pass.


In [10]:
bundle_path=work_dir/f'Heuristic_{RUN_ID}_outputs.zip'
with zipfile.ZipFile(bundle_path,'w',compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(output_dir.rglob('*')):
        if path.is_file() and '__pycache__' not in path.parts and path.suffix!='.tmp':
            archive.write(path,arcname=str(path.relative_to(output_dir)))
print('Results:',bundle_path.name)
print('Package included:',package_ready)
if package_ready:print('For Kaggle, extract and upload submission.tar.gz only.')
else:print('No submission archive included. Review process logs and partial results before rerunning.')
files.download(str(bundle_path))


Results: Heuristic_20261121T232326Z_2121ee87_outputs.zip
Package included: True
For Kaggle, extract and upload submission.tar.gz only.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Results and next work

The heuristic completed the local workflow and the manual Kaggle submission. The saved evidence shows 24 wins in 30 evaluation games, 176 demonstration decisions from separate games, two successful archive tests, and zero invalid responses or engine rejections.

Keep `team_a/src/heuristic.py` and `team_a/src/heuristic_runner.py` separate from the random baseline and its adapter. The tested source is also saved under `team_a/experiments/day3/source`.

Next, agree on the feature interface with Kavya and collect more demonstration games before evaluating the learned models. Use the manifest to preserve the training, validation and held out groups. `A-R1` and `A-R2` remain pending.

The recorded computational outputs come from the completed Colab run. The simulator was not rerun locally after the explanatory text was edited.

References: [simulator API](https://matsuoinstitute.github.io/cabt/api.html) and [competition](https://www.kaggle.com/competitions/the-pokemon-company-ptcg-ai-battle-challenge-playground).
